<a href="https://colab.research.google.com/github/Grenki-with-cheese/dissertation-notebook-2213935-cn6000/blob/main/notebook03_Erasing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Drive mount without torch to avoid conflicts

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

Mounted at /content/drive
Drive mounted.


ESD installation following the readme.md (Gondikota et al. 2023)
It will conflict

In [2]:
!git clone https://github.com/rohitgandikota/erasing.git /content/erasing
%cd /content/erasing
#conda lines removed because running on colab instead of local environment
!pip install -r requirements.txt -q

!pip freeze>/content/drive/MyDrive/MyDissertationCN6000/requirements_esd.txt
print("ESD installed")
print("Restart session, rerun cell 1 and skip to cell 3")

Cloning into '/content/erasing'...
remote: Enumerating objects: 410, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 410 (delta 107), reused 75 (delta 75), pack-reused 279 (from 1)
Receiving objects: 100% (410/410), 10.97 MiB | 35.54 MiB/s, done.
Resolving deltas: 100% (179/179), done.
/content/erasing
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 139.0 MB/s eta 0:0

In [3]:
import os
PROJECT_ROOT = '/content/drive/MyDrive/MyDissertationCN6000'
os.chdir('/content/erasing') #changed direction from root to esd specific

#silence HF download progress bars to prevent the GitHub-render widget bug
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import torch
gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

#bfloat16 (hardcoded in the script) requires Ampere or newer
assert any(x in gpu_name for x in ["A100", "L4", "H100", "A10"]), (
    f"GPU {gpu_name} may not support bfloat16. Need A100/L4/H100/A10."
)

#confirm the ESD env is in effect
import diffusers, transformers
print(f"diffusers: {diffusers.__version__}  (expect 0.37.x)")
print(f"transformers: {transformers.__version__}  (expect 5.x)")
print(f"torch: {torch.__version__}  (expect 2.11.x)")

GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
diffusers: 0.37.1  (expect 0.37.x)
transformers: 5.3.0  (expect 5.x)
torch: 2.11.0+cu130  (expect 2.11.x)


In [4]:
!pip uninstall torch_xla -y #to run before esd training

Found existing installation: torch-xla 2.9.0
Uninstalling torch-xla-2.9.0:
  Successfully uninstalled torch-xla-2.9.0


Running ESD training

In [5]:
import datetime
date_str = datetime.date.today().isoformat()
log_path = f"/content/drive/MyDrive/MyDissertationCN6000/logs/training_log_{date_str}.txt"

!mkdir -p /content/drive/MyDrive/MyDissertationCN6000/logs

!python esd_sd.py \
    --erase_concept "Vincent van Gogh" \
    --train_method "esd-x" \
    --iterations 1000 \
    --negative_guidance 2 2>&1 | tee "{log_path}"

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Training ESD (sd): 100%|██████████| 1000/1000 [19:34<00:00,  1.17s/it, esd_loss=0.0013, timestep=39]
Saved checkpoint to esd-models/sd/esd-Vincent_van_Gogh-from-Vincent_van_Gogh-esdx.safetensors


In [6]:
import shutil

shutil.copy(
    "/content/erasing/esd-models/sd/esd-Vincent_van_Gogh-from-Vincent_van_Gogh-esdx.safetensors",
    "/content/drive/MyDrive/MyDissertationCN6000/checkpoints/esd_vangogh_test.safetensors",
)
print("Copied.")

Copied.
